# 01 — Baseline
**Entrega 1.** Naive forecast y media móvil. Referencia de error mínimo esperado.

In [5]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, ".")
from utils import load_data, compute_rmse, make_submission, train_val_split, INDEX_COLS
data = load_data()
train_full = data["train_indices"][INDEX_COLS]
test_dates = data["test_dates"].index
train, val = train_val_split(train_full, val_size=252)
print(f"Train: {train.shape}  Val: {val.shape}  Test dates: {len(test_dates)}")


Train: (2095, 6)  Val: (252, 6)  Test dates: 262


## Estrategias de baseline

In [6]:
def naive_forecast(last_values, dates):
    return pd.DataFrame({c: [last_values[c]]*len(dates) for c in INDEX_COLS}, index=dates)

def rolling_mean_forecast(train, dates, window=20):
    m = train[INDEX_COLS].tail(window).mean()
    return pd.DataFrame({c: [m[c]]*len(dates) for c in INDEX_COLS}, index=dates)

def exp_smooth_forecast(train, dates, alpha=0.3):
    preds = {}
    for col in INDEX_COLS:
        s = train[col].values
        level = s[0]
        for v in s[1:]: level = alpha * v + (1-alpha) * level
        preds[col] = [level] * len(dates)
    return pd.DataFrame(preds, index=dates)


## Validación local

In [7]:
for name, fn in [("naive", lambda: naive_forecast(train.iloc[-1], val.index)),
                 ("rolling_20", lambda: rolling_mean_forecast(train, val.index, 20)),
                 ("exp_smooth", lambda: exp_smooth_forecast(train, val.index))]:
    pred = fn()
    rmse = compute_rmse(val, pred)
    print(f"  [{name}]  RMSE = {rmse:,.2f}")


  [naive]  RMSE = 3,276.24
  [rolling_20]  RMSE = 3,507.41
  [exp_smooth]  RMSE = 3,438.25


## Generar submission

In [8]:
pred_submission = rolling_mean_forecast(train_full, test_dates, window=20)
path = make_submission(pred_submission, "submission_01_baseline.csv")
pred_submission.head()


Submission saved: c:\Users\1jose\Desktop\previa hackatlon\hackathon\submissions\submission_01_baseline.csv


,Index_A,Index_B,Index_C,Index_D,Index_E,Index_F
Date,,,,,,
2024-01-01,16491.496143,4693.051489,20.323402,16355.866158,126.534521,42926.100586
2024-01-02,16491.496143,4693.051489,20.323402,16355.866158,126.534521,42926.100586
2024-01-03,16491.496143,4693.051489,20.323402,16355.866158,126.534521,42926.100586
2024-01-04,16491.496143,4693.051489,20.323402,16355.866158,126.534521,42926.100586
2024-01-05,16491.496143,4693.051489,20.323402,16355.866158,126.534521,42926.100586
